In [4]:
import pandas as pd 
import sqlite3
con=sqlite3.connect('../data/raw/ipl.db')
def q(sql):
    return pd.read_sql(sql,con)

In [5]:
q("""
SELECT bowler_type
FROM deliveries;
""")

,bowler_type
0,Right arm Medium
1,Right arm Medium
2,Right arm Medium
3,Right arm Medium
4,Right arm Medium
...,...
288221,Right arm Fast
288222,Right arm Fast
288223,Right arm Fast
288224,Right arm Medium


find

In [6]:
q("""
SELECT bowler_type ,LENGTH(bowler_type) AS len,COUNT(*) AS rows
FROM deliveries
GROUP BY bowler_type
ORDER BY len
LIMIT 5;
""")

,bowler_type,len,rows
0,NaN,NaN,339
1,,1.0,8
2,Legbreak,8.0,13893
3,Left arm Fast,13.0,3727
4,Right arm Fast,14.0,42126


fx

In [7]:
q("""
SELECT bowler_type,LENGTH(bowler_type) AS old_length,NULLIF(TRIM(bowler_type),'') AS bowler_type_clean
FROM deliveries
WHERE TRIM (bowler_type)='';
""")

,bowler_type,old_length,bowler_type_clean
0,,1,None
1,,1,None
2,,1,None
3,,1,None
4,,1,None
5,,1,None
6,,1,None
7,,1,None


verify

In [8]:
q("""
SELECT COUNT(*) AS whitespaces_only
FROM 
(SELECT NULLIF(TRIM(bowler_type),'') AS bowler_type FROM deliveries)
WHERE bowler_type IS NOT NULL AND TRIM(bowler_type)='';
""")

,whitespaces_only
0,0


In [9]:
q("""
SELECT COUNT(*) AS unknown_bowler_type
FROM deliveries
WHERE NULLIF(TRIM(bowler_type),'') IS NULL;""")

,unknown_bowler_type
0,347


In [10]:
q("""
SELECT DISTINCT(bowler_type)
FROM deliveries;""")

,bowler_type
0,Right arm Medium
1,Left arm Fast medium
2,Right arm Fast medium
3,Slow Left arm Orthodox
4,Legbreak Googly
5,Right arm Medium fast
6,Right arm Fast
7,Left arm Fast
8,Left arm Medium
9,Left arm Medium fast


In [11]:
q("""
SELECT toss_decision,
CASE WHEN toss_decision='bat' THEN 'Batting First'
WHEN toss_decision='field' THEN 'Feilding First'
ELSE toss_decision END AS toss_decision_clean,
COUNT(*)
FROM matches
GROUP BY toss_decision;""")

,toss_decision,toss_decision_clean,COUNT(*)
0,bat,Batting First,412
1,field,Feilding First,800


find

In [12]:
q("""
SELECT bowler_type,COUNT(*) AS n
FROM deliveries
GROUP BY bowler_type
ORDER BY bowler_type;""")

,bowler_type,n
0,NaN,339
1,,8
2,Left arm Fast,3727
3,Left arm Fast medium,15376
4,Left arm Medium,8778
5,Left arm Medium fast,13495
6,Left arm Wrist spin,4279
7,Legbreak,13893
8,Legbreak Googly,22670
9,Right arm Fast,42126


fix

In [13]:
q("""
SELECT bowler_type,
CASE WHEN bowler_type='Right arm Fast Medium' THEN 'Right arm Fast medium'
ELSE bowler_type END AS bowler_type_clean
FROM deliveries;""")

,bowler_type,bowler_type_clean
0,Right arm Medium,Right arm Medium
1,Right arm Medium,Right arm Medium
2,Right arm Medium,Right arm Medium
3,Right arm Medium,Right arm Medium
4,Right arm Medium,Right arm Medium
...,...,...
288221,Right arm Fast,Right arm Fast
288222,Right arm Fast,Right arm Fast
288223,Right arm Fast,Right arm Fast
288224,Right arm Medium,Right arm Medium


combine


In [14]:
q("""
SELECT 
CASE WHEN NULLIF(TRIM(bowler_type),'')='Right arm Fast Medium' THEN 'Right  arm Fast medium'
ELSE NULLIF(TRIM(bowler_type),'') END AS bowler_type_clean
FROM deliveries;""")

,bowler_type_clean
0,Right arm Medium
1,Right arm Medium
2,Right arm Medium
3,Right arm Medium
4,Right arm Medium
...,...
288221,Right arm Fast
288222,Right arm Fast
288223,Right arm Fast
288224,Right arm Medium


In [15]:
q("""
SELECT COUNT (DISTINCT bowler_type) AS n
FROM deliveries;""")

,n
0,21


verify

In [16]:
q("""
SELECT COUNT (DISTINCT 
CASE WHEN NULLIF(TRIM(bowler_type),'')='Right arm Fast Medium' THEN 'Right arm Fast medium'
ELSE NULLIF(TRIM(bowler_type),'') END )AS cleaned_count
FROM deliveries;""")

,cleaned_count
0,19


team-team name 

find


In [17]:
q("""
SELECT team_name,
(SELECT COUNT(*) FROM matches m
WHERE m.team1=t.team_name OR m.team2=t.team_name) AS played
FROM teams t
ORDER BY played;""")

,team_name,played
0,Deccan Chargers,0
1,Rising Pune Supergiants,0
2,Kochi Tuskers Kerala,14
3,Rising Pune Supergiant,30
4,Gujarat Lions,30
5,Pune Warriors,46
6,Lucknow Super Giants,66
7,Gujarat Titans,69
8,Rajasthan Royals,245
9,Chennai Super Kings,260


fix and verified

In [18]:
q("""
SELECT team_name,
CASE WHEN team_name='Rising Pune Supergiants' THEN 'Rising Pune  Supergiant'
ELSE team_name END AS team_name_clean
FROM teams;""")

,team_name,team_name_clean
0,Royal Challengers Bangalore,Royal Challengers Bangalore
1,Sunrisers Hyderabad,Sunrisers Hyderabad
2,Mumbai Indians,Mumbai Indians
3,Rising Pune Supergiant,Rising Pune Supergiant
4,Gujarat Lions,Gujarat Lions
5,Kolkata Knight Riders,Kolkata Knight Riders
6,Chennai Super Kings,Chennai Super Kings
7,Rajasthan Royals,Rajasthan Royals
8,Delhi Capitals,Delhi Capitals
9,Punjab Kings,Punjab Kings


In [19]:
q("""
 SELECT INSTR('Wankhede Stadium,Mumbai',',') AS coma_position""")

,coma_position
0,17


In [20]:
q("""
SELECT SUBSTR('Wankhede Stadium,Mumbai',1,17-1) AS extract""")

,extract
0,Wankhede Stadium


In [21]:
q(""" SELECT COUNT (DISTINCT venue) 
FROM venues;""")

,COUNT (DISTINCT venue)
0,59


In [22]:
q(""" SELECT  DISTINCT venue
FROM venues;""")

,venue
0,M Chinnaswamy Stadium
1,"Rajiv Gandhi International Stadium, Uppal"
2,Maharashtra Cricket Association Stadium
3,Saurashtra Cricket Association Stadium
4,Holkar Cricket Stadium
5,M.Chinnaswamy Stadium
6,Wankhede Stadium
7,Eden Gardens
8,Feroz Shah Kotla
9,"Punjab Cricket Association IS Bindra Stadium, ..."


In [23]:
q(""" 
SELECT COUNT(DISTINCT TRIM(SUBSTR(venue, 1,
CASE WHEN INSTR(venue,',') > 0
THEN INSTR(venue,',')-1 ELSE LENGTH(venue)END))) AS after_rule
FROM matches;
""")

,after_rule
0,42


In [24]:
q(""" 
SELECT DISTINCT TRIM(SUBSTR(venue, 1,
CASE WHEN INSTR(venue,',') > 0
THEN INSTR(venue,',')-1 ELSE LENGTH(venue)END)) AS venue_clean
FROM matches;
""")

,venue_clean
0,M Chinnaswamy Stadium
1,Rajiv Gandhi International Stadium
2,Maharashtra Cricket Association Stadium
3,Saurashtra Cricket Association Stadium
4,Holkar Cricket Stadium
5,M.Chinnaswamy Stadium
6,Wankhede Stadium
7,Eden Gardens
8,Feroz Shah Kotla
9,Punjab Cricket Association IS Bindra Stadium


Find

In [25]:
q("""SELECT COUNT(*) AS total, 
SUM(CASE WHEN city IS NULL THEN 1 ELSE 0 END) AS missing_city 
FROM matches;  """)

,total,missing_city
0,1212,51


In [39]:
q("""SELECT COALESCE(city, 'UNKNOWN') AS city_filled 
FROM matches; """)

,city_filled
0,Bangalore
1,Hyderabad
2,Pune
3,Rajkot
4,Indore
...,...
1207,Delhi
1208,New Chandigarh
1209,Mumbai
1210,Ahmedabad


In [26]:
q("SELECT COALESCE('KOCHI', 'UNKNOWN');")

,"COALESCE('KOCHI', 'UNKNOWN')"
0,KOCHI


In [27]:
q("SELECT COALESCE(NULL, 'UNKNOWN');")

,"COALESCE(NULL, 'UNKNOWN')"
0,UNKNOWN


In [28]:
q(""" 
SELECT city,COALESCE(city,'UNKNOWN')
FROM matches
WHERE city IS NULL
LIMIT 10;
""")

,city,"COALESCE(city,'UNKNOWN')"
0,None,UNKNOWN
1,None,UNKNOWN
2,None,UNKNOWN
3,None,UNKNOWN
4,None,UNKNOWN
5,None,UNKNOWN
6,None,UNKNOWN
7,None,UNKNOWN
8,None,UNKNOWN
9,None,UNKNOWN


In [29]:
q(""" SELECT COUNT(*) AS N
  FROM matches
  WHERE COALESCE (city,'UNKNOWN') = 'UNKNOWN';""")

,N
0,51


In [30]:
q(" SELECT CAST ('2019' AS INTEGER);")

,CAST ('2019' AS INTEGER)
0,2019


In [31]:
q('''
SELECT DISTINCT season,
SUBSTR(season,1,4)AS first_four,
CAST(SUBSTR(season,1,4)AS INTEGER)AS season_year
FROM matches
ORDER BY season_year;
''')

,season,first_four,season_year
0,2008,2008,2008
1,2009,2009,2009
2,2010,2010,2010
3,2011,2011,2011
4,2012,2012,2012
5,2013,2013,2013
6,2014,2014,2014
7,2015,2015,2015
8,2016,2016,2016
9,2017,2017,2017


In [32]:
q("""SELECT venue, COUNT(*) AS n
FROM venues
GROUP BY venue;""")

,venue,n
0,Arun Jaitley Stadium,1
1,"Arun Jaitley Stadium, Delhi",1
2,Barabati Stadium,1
3,"Barsapara Cricket Stadium, Guwahati",1
4,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,1
5,Brabourne Stadium,1
6,"Brabourne Stadium, Mumbai",1
7,Buffalo Park,1
8,De Beers Diamond Oval,1
9,Dr DY Patil Sports Academy,1


In [33]:
q("""
SELECT venue, COUNT(*) 
OVER(PARTITION BY venue)
FROM venues;
""")

,venue,COUNT(*) \nOVER(PARTITION BY venue)
0,Arun Jaitley Stadium,1
1,"Arun Jaitley Stadium, Delhi",1
2,Barabati Stadium,1
3,"Barsapara Cricket Stadium, Guwahati",1
4,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,1
...,...,...
58,SuperSport Park,1
59,"Vidarbha Cricket Association Stadium, Jamtha",1
60,Wankhede Stadium,1
61,"Wankhede Stadium, Mumbai",1


In [34]:
q("""
SELECT venue_id,venue,
ROW_NUMBER()OVER(PARTITION BY venue ORDER BY venue_id) AS rn
FROM venues;
""") 

,venue_id,venue,rn
0,16,Arun Jaitley Stadium,1
1,24,"Arun Jaitley Stadium, Delhi",1
2,55,Barabati Stadium,1
3,37,"Barsapara Cricket Stadium, Guwahati",1
4,34,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,1
...,...,...,...
58,48,SuperSport Park,1
59,56,"Vidarbha Cricket Association Stadium, Jamtha",1
60,7,Wankhede Stadium,1
61,22,"Wankhede Stadium, Mumbai",1


verify

In [35]:
q("""
WITH ranked AS(
SELECT venue_id,venue,
ROW_NUMBER()OVER(PARTITION BY venue ORDER BY venue_id) AS rn
FROM venues)
SELECT COUNT(*) AS kept
FROM ranked
WHERE rn=1;""")

,kept
0,59


CREATE VIEW - to save the query

In [36]:
con.executescript("""
DROP VIEW IF EXISTS v_matches_clean;
CREATE VIEW v_matches_clean AS 
SELECT *,
CASE WHEN venue_comma_clean='M.Chinnaswamy Stadium'THEN 'M Chinnaswamy Stadium' ELSE venue_comma_clean END AS venue_clean
FROM(SELECT *,
TRIM(SUBSTR(venue,1,CASE 
WHEN INSTR(venue,',')>0 
THEN INSTR(venue,',')-1 
ELSE LENGTH(venue)
END)) AS venue_comma_clean
FROM matches); """)
con.commit()

In [37]:
q("""
SELECT COUNT(DISTINCT venue_clean)
FROM v_matches_clean;""")

,COUNT(DISTINCT venue_clean)
0,41


In [38]:
q("""
SELECT venue_clean,COUNT(*) AS matches_hosted
FROM v_matches_clean
GROUP BY venue_clean
ORDER BY matches_hosted DESC
LIMIT 5;""")

,venue_clean,matches_hosted
0,Wankhede Stadium,130
1,M Chinnaswamy Stadium,104
2,Eden Gardens,104
3,MA Chidambaram Stadium,95
4,Rajiv Gandhi International Stadium,87
